## Behandlung der Variable: Analyse der Zeitabstände (`time_gap`)

In diesem Abschnitt widmen wir uns der zentralen Zielvariable unseres Modells: den Zeitabständen der Fahrer zum Etappensieger (`time_gap`). Im Radsport ist die Modellierung dieser Variable eine besondere Herausforderung, da sie zwei fundamentale Eigenschaften aufweist:

1. **Reine Rechtsschiefe (Extreme Skewness):** Der Etappensieger hat einen Abstand von `0` Sekunden. Ein Großteil des Hauptfeldes (Peloton) kommt oft mit demselben oder einem geringen Zeitabstand im Ziel an. Danach reißt das Feld ab, und es bildet sich ein langer "Schwanz" (Long Tail) von Fahrern mit massiven Zeitabständen (z. B. das *Gruppetto* bei Bergetappen).
2. **Ausreißer vs. sportliche Realität:** Sehr große Abweichungen (Stundencharakter bei schweren Rundfahrten) sind statistisch gesehen extreme Ausreißer, spiegeln jedoch die reale Dynamik des Rennens wider.

### Ziel dieser Analyse:
* **Verteilungsprüfung:** Mathematische Bestimmung der Schiefe und Wölbung mittels `scipy.stats`.
* **Umgang mit NaNs / DNF:** Definition von Strategien für Fahrer, die das Ziel nicht erreicht haben (Did Not Finish).
* **Transformationen:** Evaluierung von Methoden (z. B. Log-Transformation oder Box-Cox), um die Rechtsschiefe für Lukas' XGBRanker oder Regressionsmodelle zu normalisieren.

In [ ]:
import os
import numpy as np
import pandas as pd


In [3]:
# DF Laden
path = "../../data/processed/11_cleaned_master_data.pkl"

df = pd.read_pickle(path)

In [4]:
print(f"Shape: {df.shape[0]:,} Zeilen x {df.shape[1]} Spalten")

Shape: 222,818 Zeilen x 48 Spalten


In [5]:
# 1. Basis-Statistik
print("--- Analyse der Zeitabstände ---")
print(df['time_gap'].describe())

# 2. NaNs und Nuller zählen
print(f"\nFehlende Werte (NaNs): {df['time_gap'].isna().sum():,}")


# 3. Schreibweise und Datentyp prüfen
print(df['time_gap'].dtype)
print()
print(df['time_gap'].head(20))

--- Analyse der Zeitabstände ---
count     222818
unique      6747
top       ,,0:00
freq       39510
Name: time_gap, dtype: object

Fehlende Werte (NaNs): 0
object

0     20:5120:51
1       0:020:02
2       0:530:53
3       0:570:57
4       0:590:59
5       1:021:02
6         ,,1:02
7       1:041:04
8       1:051:05
9       1:061:06
10      1:071:07
11      1:081:08
12      1:121:12
13      1:131:13
14        ,,1:13
15      1:161:16
16      1:181:18
17      1:241:24
18      1:251:25
19      1:261:26
Name: time_gap, dtype: object


In [13]:
# Anzahl fehlender Werte im gesamten DataFrame pro Spalte
print(df.isna().sum())
print("--------")
# Anzahl fehlender Werte speziell in der Spalte 'rank'
print(df['rank'].isna().sum())
print("\n\n")

# Na Zeilen Rank ausgeben
na_rows = df[df['rank'].isna()]
print(na_rows[['rank', 'time_gap']].head(10))

race                                0
year                                0
url                                 0
rank                             2785
rider_url                           0
time_gap                            0
birthdate                           0
height                           1058
name                                0
nationality                         0
weight                           1058
url_name                            0
departure                           0
arrival                             0
distance                            0
vertical_meters                  5235
profile_score                    5235
won_how                             0
avg_speed                         140
race_ranking                        0
one_day_races                       0
gc                                  0
time_trial                          0
sprint                              0
climber                             0
hills                               0
stage_nr    

### Droppen der Rank na Zeilen

In [14]:
print("Vorher:", len(df))
df = df.dropna(subset=['rank']).copy()
print("Nachher:", len(df))
print("Verbleibende NaNs in 'rank':", df['rank'].isna().sum())

Vorher: 222818
Nachher: 220033
Verbleibende NaNs in 'rank': 0


# Aufbau der Spalte

- Siegerzeit ganz oben "4:30:074:30:07"
- danach Gap zum Vordermann "0:070:07"
- wenn zu gleicher Zeit mit einem anderen ",,0:07"
- zudem Sonderwertungen wie "DNF", "DNS", "DSQ", "OTL", "HD" -> bereits eliminiert
- Problem der "**4:30:07**4:30:07" 

```json
{
  "race": "giro-d-italia",
  "year": 2009,
  "url": "https://www.procyclingstats.com/race/giro-d-italia/2009/stage-20",
  "metadata": {
    "departure": "Napoli",
    "arrival": "Anagni",
    "distance": "203 km",
    "vertical_meters": "1577",
    "profile_score": "53",
    "won_how": "Sprint à deux",
    "avg_speed": "45.092 km/h",
    "race_ranking": "n/a"
  },
  "results": [
    {
      "rank": "1",
      "rider_url": "philippe-gilbert",
      "time_gap": "4:30:074:30:07"
    },
    {
      "rank": "2",
      "rider_url": "thomas-voeckler",
      "time_gap": "0:020:02"
    },
    {
      "rank": "3",
      "rider_url": "stefano-garzelli",
      "time_gap": "0:070:07"
    },
    {
      "rank": "4",
      "rider_url": "allan-davis",
      "time_gap": ",,0:07"
    },
    {
      "rank": "5",
      "rider_url": "sebastien-hinault",
      "time_gap": ",,0:07"
    },
    {
      "rank": "6",
      "rider_url": "franco-pellizotti",
      "time_gap": ",,0:07"
    },
    {
      "rank": "7",
      "rider_url": "edvald-boasson-hagen",
      "time_gap": ",,0:07"
    },
    {
      "rank": "8",
      "rider_url": "giovanni-visconti",
      "time_gap": ",,0:07"
    },
    {
      "rank": "9",
      "rider_url": "simon-gerrans",
      "time_gap": ",,0:07"
    },
    {
      "rank": "10",
      "rider_url": "serge-pauwels",
      "time_gap": ",,0:07"
    },
    {
      "rank": "11",
      "rider_url": "denis-menchov",
      "time_gap": ",,0:07"
    },
    {
      "rank": "12",
      "rider_url": "danilo-di-luca",
      "time_gap": ",,0:07"
    },
    {
      "rank": "13",
      "rider_url": "rubens-bertogliati",
      "time_gap": ",,0:07"
    },
    {
      "rank": "14",
      "rider_url": "ruggero-marzoli",
      "time_gap": ",,0:07"
    },
{..................................}
    {
      "rank": "141",
      "rider_url": "bradley-wiggins",
      "time_gap": ",,7:55"
    },
    {
      "rank": "142",
      "rider_url": "olivier-kaisen",
      "time_gap": "8:248:24"
    },
    {
      "rank": "143",
      "rider_url": "jure-golcer",
      "time_gap": "8:448:44"
    },
    {
      "rank": "144",
      "rider_url": "francesco-de-bonis",
      "time_gap": "13:2813:28"
    },
    {
      "rank": "145",
      "rider_url": "giuseppe-palumbo",
      "time_gap": ",,13:28"
    },
{..................................}
    {
      "rank": "169",
      "rider_url": "matthieu-sprick",
      "time_gap": ",,13:28"
    },
  ]
}
```

In [17]:
print(df['time_gap'].dtype)

object


### Transformierung von time_gap

#### Bedingung finale Spalte:
- keine Textzeichen mehr
- numerischer Datentyp (float)
- glatte Sekunden hernehmen (Sieger 0.0, 1.0, 2.0, 2.0 ... 540.0...)


<class 'pandas.Series'>
Index: 220033 entries, 0 to 225538
Series name: time_gap
Non-Null Count   Dtype 
--------------   ----- 
220033 non-null  object
dtypes: object(1)
memory usage: 3.4+ MB
None


In [20]:
def parse_time_gap_to_int(val):
    if pd.isna(val) or val == "n/a" or val == "":
        return np.nan

    # Kommas und Leerzeichen entfernen
    val = str(val).strip().replace(",", "")
    if not val:
        return np.nan

    # Verdopplung halbieren (z.B. "8:248:24" -> "8:24")
    half = len(val) // 2
    if len(val) % 2 == 0 and val[:half] == val[half:]:
        val = val[:half]

    # In Sekunden umrechnen
    try:
        parts = val.split(":")
        if len(parts) == 3:  # HH:MM:SS
            return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
        elif len(parts) == 2:  # MM:SS
            return int(parts[0]) * 60 + int(parts[1])
        else:  # Reine Sekundenzahl
            return int(float(val))
    except:
        return np.nan


# 1. Spalte anwenden
df["time_gap_seconds"] = df["time_gap"].apply(parse_time_gap_to_int)

# 2. Sieger (Rank 1) auf 0 Sekunden setzen
df["rank_str"] = df["rank"].astype(str).str.strip()
df.loc[
    df["rank_str"].str.startswith("1.") | (df["rank_str"] == "1"),
    "time_gap_seconds",
    ] = 0
df.drop(columns=["rank_str"], inplace=True)

# 3. In glatte Integers (mit NaN-Support) konvertieren
df["time_gap_seconds"] = df["time_gap_seconds"].astype("Int64")

# Kontroll-Check für Datentyp und Inhalt
print(df[["rank", "time_gap", "time_gap_seconds"]].head(15))
print("\nSpalten-Typ:", df["time_gap_seconds"].dtype)
print("\nNAs in time_gap_seconds: ", df['time_gap_seconds'].isna().sum())

    rank    time_gap  time_gap_seconds
0    1.0  20:5120:51                 0
1    2.0    0:020:02                 2
2    3.0    0:530:53                53
3    4.0    0:570:57                57
4    5.0    0:590:59                59
5    6.0    1:021:02                62
6    7.0      ,,1:02                62
7    8.0    1:041:04                64
8    9.0    1:051:05                65
9   10.0    1:061:06                66
10  11.0    1:071:07                67
11  12.0    1:081:08                68
12  13.0    1:121:12                72
13  14.0    1:131:13                73
14  15.0      ,,1:13                73

Spalten-Typ: Int64

NAs in time_gap_seconds:  4769


In [ ]:
Immer noch einige NA Werte nach dem Transfomrationsversuch

Prüfen warum es schief lief

In [ ]:
# Verdacht liegt auf Time Trial, da es die absoluten Zeiten nimmt.

nan_won_how = df[df['time_gap_seconds'].isna()]['won_how'].value_counts(dropna=False)
print(nan_won_how)

# Ausgabe Time Trial Zeilen
time_trial_nas = df[
    (df['time_gap_seconds'].isna()) &
    (df['won_how'].astype(str).str.lower().str.contains('trial'))
]

print(time_trial_nas[['race', 'year', 'rank', 'won_how', 'time_gap', 'time_gap_seconds']].head(15))

# Verteilung welche Ränge es am meisten betrifft
nan_ranks = df[df['time_gap_seconds'].isna()]['rank'].value_counts(dropna=False)
print(nan_ranks)

print(df['rank'].dtype)

won_how
Time trial               2764
Sprint of large group     992
Sprint of small group     404
-                         199
Sprint à deux             135
5.5 km solo                78
26.8 km solo               41
12 km solo                 38
3.7 km solo                31
Sprint of 15 riders        24
0.9 km solo                18
0.8 km solo                13
14 km solo                 11
4 km solo                   3
1.1 km solo                 2
0.7 km solo                 2
1.5 km solo                 1
5.3 km solo                 1
8.5 km solo                 1
20 km solo                  1
44 km solo                  1
1.6 km solo                 1
6.6 km solo                 1
17.2 km solo                1
0.4 km solo                 1
6.1 km solo                 1
22.3 km solo                1
13 km solo                  1
0.5 km solo                 1
Sprint of 9 riders          1
Name: count, dtype: int64
                 race  year  rank     won_how   time_gap  time_gap

In [ ]:
Somit liegt wohl ein Problem im Zeitfahren vor

- generell Frage ob Zeitfahrdaten Sinn machen
    - komplett andere Sportart
    - somit evtl. Lernen von anderer Sportart
    - zudem weitere Disziplin Team Zeitfahren vorhanden

Entscheidung: Drop Time Trial


In [32]:
print("Vorher (inkl. Zeitfahren):", len(df))

# Alle normalen Etappen behalten, Zeitfahren (Einzel & Mannschaft) ausschließen
df = df[~df['won_how'].astype(str).str.lower().str.contains('trial')].copy()

print("Nachher (nur noch Massenstarts):", len(df))

Vorher (inkl. Zeitfahren): 220033
Nachher (nur noch Massenstarts): 197561


In [38]:
print("\nNAs in time_gap_seconds: ", df['time_gap_seconds'].isna().sum())

# immer noch 2005 nas bei time-gap_seconds
# Alle verbliebenen Zeilen mit NaN in den Sekunden holen

rest_nas = df[df['time_gap_seconds'].isna()]

print("DIE HÄUFIGSTEN TEXT-WERTE IN DIESEN 2.005 ZEILEN")
print(rest_nas['time_gap'].value_counts(dropna=False).head(50))


print("BEI WELCHEN RÄNGEN PASSIERT DAS AM HÄUFIGSTEN?")
print(rest_nas['rank'].value_counts().head(20))


NAs in time_gap_seconds:  2005
DIE HÄUFIGSTEN TEXT-WERTE IN DIESEN 2.005 ZEILEN
time_gap
*,,0:00        762
*0:000:00      318
n/a            146
*,,0:40         67
*,,22:29        42
*,,0:18         41
*,,20:56        23
*,,0:07         22
*,,14:36        22
*0:070:07       19
*,,27:17        19
*,,32:04        18
*,,14:48        17
*0:010:01       13
*,,0:01         12
*0:400:40       11
*,,5:50         11
*,,19:24        11
*,,3:37         10
*,,15:28         9
*,,0:02          8
*,,13:21         8
*,,10:02         8
*1:061:06        7
*0:020:02        7
*2:382:38        7
*,,11:57         7
*,,8:07          7
*0:260:26        6
*0:040:04        6
*,,0:31          6
*,,14:24         6
*,,21:58         6
*,,19:35         5
*,,14:02         4
*,,10:49         4
*14:4814:48      4
*3:163:16        4
*2:132:13        4
*,,7:05          4
*13:1313:13      4
*0:050:05        3
*0:270:27        3
*0:090:09        3
*3:373:37        3
*13:2113:21      3
*1:491:49        3
*3:073:07        

In [39]:
#Anpassung des Parsers

def parse_time_gap_ultimate(val):
    if pd.isna(val) or val == "n/a" or val == "":
        return np.nan

    # Bereinigung von Sternchen und Kommas
    val = str(val).strip().replace("*", "").replace(",", "")

    # Wenn nach dem Putzen nichts mehr da ist oder es 0:00 war
    if not val or val == "0:00":
        return 0

    # Das Verdopplungs-Problem lösen (z.B. "0:000:00" -> "0:00" -> 0)
    half = len(val) // 2
    if len(val) % 2 == 0 and val[:half] == val[half:]:
        val = val[:half]

    if val == "0:00" or val == "0" or val == "00":
        return 0

    # In Sekunden umrechnen
    try:
        parts = val.split(":")
        if len(parts) == 3:  # HH:MM:SS
            return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
        elif len(parts) == 2:  # MM:SS
            return int(parts[0]) * 60 + int(parts[1])
        else:  # Reine Sekundenzahl
            return int(float(val))
    except:
        return np.nan


# Parser anwenden
df["time_gap_seconds"] = df["time_gap"].apply(parse_time_gap_ultimate)

# Sieger (Rank 1) auf 0 Sekunden setzen
df["rank_str"] = df["rank"].astype(str).str.strip()
df.loc[
    df["rank_str"].str.startswith("1.") | (df["rank_str"] == "1"),
    "time_gap_seconds",
] = 0
df.drop(columns=["rank_str"], inplace=True)

# In glatte Integers (mit NaN-Support) konvertieren
df["time_gap_seconds"] = df["time_gap_seconds"].astype("Int64")

# Der finale Kontroll-Check auf verbliebene NAs
print(
    "Verbleibende NAs in time_gap_seconds:",
    df["time_gap_seconds"].isna().sum(),
)

Verbleibende NAs in time_gap_seconds: 157


In [40]:
rest_nas = df[df['time_gap_seconds'].isna()]

print("DIE HÄUFIGSTEN TEXT-WERTE IN DIESEN 2.005 ZEILEN")
print(rest_nas['time_gap'].value_counts(dropna=False).head(50))


print("BEI WELCHEN RÄNGEN PASSIERT DAS AM HÄUFIGSTEN?")
print(rest_nas['rank'].value_counts().head(20))

DIE HÄUFIGSTEN TEXT-WERTE IN DIESEN 2.005 ZEILEN
time_gap
n/a                  146
24:06:00 24:06:00      1
24:44:00 24:44:00      1
25:05:00 25:05:00      1
25:36:00 25:36:00      1
27:34:00 27:34:00      1
28:05:00 28:05:00      1
29:44:00 29:44:00      1
29:50:00 29:50:00      1
31:17:00 31:17:00      1
32:44:00 32:44:00      1
36:53:00 36:53:00      1
Name: count, dtype: int64
BEI WELCHEN RÄNGEN PASSIERT DAS AM HÄUFIGSTEN?
rank
33.0    2
34.0    2
35.0    2
36.0    2
44.0    2
45.0    2
52.0    2
54.0    2
55.0    2
57.0    2
58.0    2
2.0     1
3.0     1
4.0     1
5.0     1
6.0     1
7.0     1
8.0     1
9.0     1
10.0    1
Name: count, dtype: int64


Entscheidung droppen der letzten 157 NA Zeilen 146 sind sowieso NA

In [41]:
# Die letzten 157 Zeilen ohne gültigen Zeitwert endgültig entfernen
print("Zeilen vor Drop", len(df))
df = df.dropna(subset=['time_gap_seconds']).copy()

print(f"Verbleibende Zeilen im Datensatz: {len(df)}")
print(f"NaNs in time_gap_seconds: {df['time_gap_seconds'].isna().sum()}")
print(f"Datentyp der Spalte: {df['time_gap_seconds'].dtype}")

Zeilen vor Drop 197561
Verbleibende Zeilen im Datensatz: 197404
NaNs in time_gap_seconds: 0
Datentyp der Spalte: Int64


### Speed Outlier erneut prüfen
Da wir Time Trial herausgenommen haben können wir gleich noch einmal auf die Speed Outlier zurückommen.
Dort hatten wir einige Ausreiser

In [ ]:
df["avg_speed"].describe()

In [ ]:
def iqr_outliers(df: pd.DataFrame, cols: list[str], k: float = 1.5) -> pd.DataFrame:
    rows = []
    for col in cols:
        s = df[col].dropna()
        if s.empty:
            continue
        q1, q3 = np.percentile(s, [25, 75])
        iqr = q3 - q1
        lower = q1 - k * iqr
        upper = q3 + k * iqr
        outliers = s[(s < lower) | (s > upper)]
        rows.append(
            {
                "column": col,
                "n": int(s.shape[0]),
                "q1": q1,
                "q3": q3,
                "iqr": iqr,
                "lower_bound": lower,
                "upper_bound": upper,
                "min": s.min(),
                "max": s.max(),
                "n_outliers": int(outliers.shape[0]),
                "outlier_pct": round(100 * outliers.shape[0] / s.shape[0], 2),
            }
        )
    return (
        pd.DataFrame(rows)
        .set_index("column")
        .sort_values("outlier_pct", ascending=False)
    )


outlier_table = iqr_outliers(df, ["avg_speed"]).round(3)
outlier_table

Keine Ausreiser vorhanden.
Geringster Wert bei rund 27 km/h auf einer kurzen Bergetappe im Jahr 2018 bei der TdF.

Im weiteren werden wir das Feature voraussichtlich noch eliminieren, da es sich dabei immer um die Avg. Speed des Siegers handelt und somit nichts für ein Modell beiträgt.

### Checkpoint

Speichern des neuen df's als 12_cleaned_master_data.pkl

In [43]:
pfad = '../../data/processed/12_cleaned_master_data.pkl'
df.to_pickle(pfad)